# Omni ChromHMM Analysis

Analysis, cross-segmentation comparison and inter-dataset summary plots.

Run the Snakemake pipeline first to produce the segmentations (`{ds}/.done`); then run
this notebook top-to-bottom:
1. **Per-segmentation analysis** — `analyze.run_analyze` / `analyze_peaks.run_analyze_peaks`
2. **Cross-segmentation comparison** — `compare.run_compare` / `compare_methods.run_compare_methods`
3. **Inter-dataset comparison & summary plots** — `compare`, `compare_out`, `summary_plots`, `emission_similarity`
4. **Results** — every plot displayed inline, grouped into five sections:
   (1) peaks number and lengths, (2) segmentation states number,
   (3) segmentation states lengths, (4) segmentation compositions,
   and (5) all other analyses.

In [ ]:
import os
import sys
import glob
import yaml
import pandas as pd
from IPython.display import Image, display, HTML

# Load configuration
config_path = os.path.abspath(os.path.expanduser("~/work/omni-chromhmm/config.yaml"))
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

project_root = os.path.dirname(config_path)
scripts_dir = os.path.join(project_root, "scripts", "analysis")
workdir = os.path.expanduser(config.get("workdir", "."))

# Import the analysis methods directly (no CLI / subprocess).
sys.path.insert(0, scripts_dir)
import importlib
import analyze
import analyze_peaks
import compare
import compare_methods
import compare_inter_dataset
import emission_similarity
import summary_plots

# Re-import the analysis modules so edits to scripts/analysis/*.py are picked up when this
# cell is re-run, without needing a kernel restart (plain `import` caches modules).
for _m in (analyze, analyze_peaks, compare, compare_methods,
           compare_inter_dataset, emission_similarity, summary_plots):
    importlib.reload(_m)

# Run everything relative to the pipeline working directory.
os.chdir(workdir)
print(f"Project root: {project_root}")
print(f"Scripts dir : {scripts_dir}")
print(f"Working dir : {workdir}")

# --- Parameters (mirrors the Snakefile) ----------------------------------
P = config["params"]
TOOLS = config["tools"]
DATASETS = config["datasets"]
MARKS = P["marks"]
CHROMHMM_BIN = P["chromhmm_bin"]
OMNI_BIN = P["omni_bin"]
HOMER_BIN = P["homer_bin"]
MACS2_BIN = P["macs2_bin"]
NSTATES = P["n_states"]
MATCH_METHOD = P.get("match_method", "comb")
CALLER_BIN = {"omni": OMNI_BIN, "homer": HOMER_BIN, "macs2": MACS2_BIN}

DO_CHROMHMM_PEAKS = P.get("chromhmm_peaks", False)
DO_REPLICATES = P.get("replicates", False)

# Peak callers to include. Edit to match the segmentations you actually produced;
# missing files are skipped gracefully throughout the notebook.
CALLERS = ["homer", "macs2", "omni"]

COORDS_DIR = os.path.join(workdir, TOOLS["coords_dir"])
GENCODE_GTF = os.path.join(workdir, TOOLS["gencode_gtf"])
MARKUPS_DIR = os.path.join(project_root, "markups")

# De-novo methods compared across datasets
INTER_DS_METHODS = (
    ["chromhmm_default"]
    + ([f"chromhmm_{c}" for c in CALLERS] if DO_CHROMHMM_PEAKS else [])
    + [f"kmeans_{c}" for c in CALLERS]
)
CHIP_DATASETS = [d for d in DATASETS if not d.endswith("_mint")]
MINT_DATASETS = [d for d in DATASETS if d.endswith("_mint")]
REP_DATASETS = [d for d in DATASETS if DO_REPLICATES and DATASETS[d].get("replicates")]


# --- Path helpers (mirror the Snakefile functions) -----------------------
def ds_of(folder):
    return folder.split("/")[0]

def folders_of(ds):
    fl = [ds]
    if DO_REPLICATES and DATASETS[ds].get("replicates"):
        fl += [f"{ds}/rep1", f"{ds}/rep2"]
    return fl

def ref_bed_path(ds):
    return f"{ds}/{DATASETS[ds]['ref_chromhmm']}_chromhmm.bed"

def seg_bin(path):
    for caller, size in CALLER_BIN.items():
        if f"/{caller}/" in path:
            return size
    return CHROMHMM_BIN

def inter_ds_bed(ds, method):
    cell = DATASETS[ds]["cell"]
    sfx = f"{MATCH_METHOD}_matched"
    if method == "chromhmm_default":
        return f"{ds}/chromhmm_default_result/{cell}_{NSTATES}_dense_{sfx}.bed"
    model, caller = method.split("_")  # chromhmm|kmeans , omni|homer|macs2
    if model == "chromhmm":
        return f"{ds}/{caller}/chromhmm_result/{caller}_{cell}_{NSTATES}_dense_{sfx}.bed"
    return f"{ds}/{caller}/{caller}_kmeans_states_{sfx}.bed"

def existing(paths):
    """Keep only paths that exist on disk (skip segmentations not produced)."""
    return [p for p in paths if os.path.exists(p)]

print(f"Callers       : {CALLERS}")
print(f"Match variant : {MATCH_METHOD}")
print(f"Inter methods : {INTER_DS_METHODS}")


## Define Analysis Functions
These functions wrap the execution of `analyze.py` and `analyze_peaks.py`.


In [ ]:
def run_analyze(seg, bin_size, outdir, inputs=None, bw_emissions=None,
                rnaseq=None, gtf=None, annotations=None, emissions_only=False):
    """Direct call into analyze.run_analyze; silently skips missing segmentations.

    Glob patterns in `inputs` are expanded inside analyze.run_analyze.
    """
    if not os.path.exists(seg):
        return
    print(f"  -> {outdir}")
    analyze.run_analyze(
        seg=seg, bin_size=bin_size, outdir=outdir,
        inputs=inputs, annotations=annotations,
        bw_emissions=bw_emissions if (bw_emissions and os.path.exists(bw_emissions)) else None,
        rnaseq=rnaseq if (rnaseq and os.path.exists(rnaseq)) else None,
        gtf=gtf if (gtf and os.path.exists(gtf)) else None,
        emissions_only=emissions_only,
    )


def run_analyze_peaks(ds, cell, marks, omni_bin, chromhmm_bin, outdir):
    """Direct call into analyze_peaks.run_analyze_peaks."""
    print(f"  -> {outdir}")
    analyze_peaks.run_analyze_peaks(
        ds=ds, cell=cell, marks=list(marks), outdir=outdir,
        omni_bin=omni_bin, chromhmm_bin=chromhmm_bin,
    )


## Run Analysis
Executing the analysis for all datasets and folders.


In [ ]:
annotations = glob.glob(os.path.join(COORDS_DIR, "*.bed.gz"))

for ds, cfg in DATASETS.items():
    print(f"Processing dataset: {ds}")
    cell = cfg["cell"]
    folders = [ds]
    if DO_REPLICATES and cfg.get("replicates"):
        folders += [f"{ds}/rep1", f"{ds}/rep2"]

    # Peak analysis
    print(f"Analyzing peaks for {ds}...")
    run_analyze_peaks(ds, cell, MARKS, P["omni_bin"], CHROMHMM_BIN, f"{ds}/peaks")

    # RNA-seq / ATAC extra annotations if available
    ds_annotations = list(annotations)
    if cfg.get("atac"):
        ds_annotations.append(f"{ds}/atac_{cfg['atac']}.bed.gz")

    # Segmentations analysis
    for folder in folders:
        print(f"Analyzing segmentations in {folder}...")

        # Reference
        ref_bed = f"{ds}/{cfg['ref_chromhmm']}_chromhmm.bed"
        run_analyze(ref_bed, CHROMHMM_BIN, f"{folder}/analysis/ref",
                    annotations=ds_annotations,
                    bw_emissions=ref_bed.replace(".bed", ".bw_emissions.npz"))

        # Default ChromHMM
        default_seg = f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense.bed"
        run_analyze(default_seg, CHROMHMM_BIN, f"{folder}/analysis/chromhmm_default_dense",
                    inputs=[f"{folder}/chromhmm_default/*.txt"], emissions_only=True)

        for variant in ["comb", "bwem", "ovlp"]:
            matched_seg = f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense_{variant}_matched.bed"
            run_analyze(matched_seg, CHROMHMM_BIN, f"{folder}/analysis/{variant}/chromhmm_default",
                        inputs=[f"{folder}/chromhmm_default/*.txt"],
                        annotations=ds_annotations,
                        bw_emissions=matched_seg.replace(".bed", ".bw_emissions.npz"),
                        rnaseq=f"{ds}/rnaseq_{cfg['rnaseq']}.tsv" if cfg.get("rnaseq") else None,
                        gtf=GENCODE_GTF if cfg.get("rnaseq") else None)

            for caller in CALLERS:
                cbin = CALLER_BIN[caller]
                peaks_dir = f"{folder}/{caller}/chromhmm_peaks"

                # KMeans
                kmeans_seg = f"{folder}/{caller}/{caller}_kmeans_states_{variant}_matched.bed"
                run_analyze(kmeans_seg, cbin, f"{folder}/analysis/{variant}/kmeans_{caller}",
                            inputs=[f"{peaks_dir}/*.txt.gz"],
                            annotations=ds_annotations,
                            bw_emissions=kmeans_seg.replace(".bed", ".bw_emissions.npz"),
                            rnaseq=f"{ds}/rnaseq_{cfg['rnaseq']}.tsv" if cfg.get("rnaseq") else None,
                            gtf=GENCODE_GTF if cfg.get("rnaseq") else None)

                # ChromHMM Peaks
                if DO_CHROMHMM_PEAKS:
                    chromhmm_peaks_seg = f"{folder}/{caller}/chromhmm_result/{caller}_{cell}_{NSTATES}_dense_{variant}_matched.bed"
                    run_analyze(chromhmm_peaks_seg, cbin, f"{folder}/analysis/{variant}/chromhmm_{caller}",
                                inputs=[f"{peaks_dir}/*.txt.gz"],
                                annotations=ds_annotations,
                                bw_emissions=chromhmm_peaks_seg.replace(".bed", ".bw_emissions.npz"),
                                rnaseq=f"{ds}/rnaseq_{cfg['rnaseq']}.tsv" if cfg.get("rnaseq") else None,
                                gtf=GENCODE_GTF if cfg.get("rnaseq") else None)


## Cross-segmentation comparison

Per dataset: transition-matrix entropy, pairwise Cohen's
κ and Jaccard similarity, emission similarity and segment-length statistics, plus the
unified method comparison table. Requires the per-segmentation analysis above to have
run (it reads `analysis/<variant>/.../jaccard.tsv` and the `.bin_emissions.npz` files).


In [ ]:
# Cross-segmentation comparison per dataset
def compare_beds_for_folder(folder, variant):
    cell = DATASETS[ds_of(folder)]["cell"]
    sfx = f"{variant}_matched"
    beds = [f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense_{sfx}.bed"]
    for caller in CALLERS:
        if DO_CHROMHMM_PEAKS:
            beds.append(f"{folder}/{caller}/chromhmm_result/{caller}_{cell}_{NSTATES}_dense_{sfx}.bed")
        beds.append(f"{folder}/{caller}/{caller}_kmeans_states_{sfx}.bed")
    return beds

def ds_compare_segs(ds, variant):
    segs = [ref_bed_path(ds)]
    for folder in folders_of(ds):
        segs += compare_beds_for_folder(folder, variant)
    return segs

variant = MATCH_METHOD
for ds in DATASETS:
    segs = existing(ds_compare_segs(ds, variant))
    if len(segs) < 2:
        print(f"{ds}: found {len(segs)} segmentation(s) for '{variant}', skipping comparison")
        continue
    print(f"Comparing {ds}: {len(segs)} segmentations (variant={variant}) ...")
    bins = [seg_bin(p) for p in segs]
    try:
        compare.run_compare(seg=segs, bins=bins,
                            outdir=f"{ds}/comparison/{variant}",
                            analysis_dir=f"{ds}/analysis/{variant}")
        compare_methods.run_compare_methods(
            analysis_dir=f"{ds}/analysis/{variant}",
            comparison_dir=f"{ds}/comparison/{variant}",
            outdir=f"{ds}/methods/{variant}",
            ref_dir=f"{ds}/analysis")
    except Exception as e:
        print(f"  ERROR comparing {ds}: {e}")


## Inter-dataset comparison & summary plots

Compares each method across all datasets (all
pairs), aggregates a cross-dataset table, and renders the summary bar/violin plots,
reference similarity, state-composition, peak and emission-similarity plots. Requires
the per-dataset comparison above (`{ds}/methods/comb/comparison_table.tsv`,
`{ds}/analysis/comb/...`) and the ENCODE reference markups (`markups/15state/`).


In [ ]:
# Inter-dataset comparison
ds_list = list(DATASETS)
cells = [DATASETS[d]["cell"] for d in ds_list]
sp_out = "out/summary_plots"
os.makedirs(sp_out, exist_ok=True)

def _try(label, fn):
    """Run one plotting step; report and continue on failure (e.g. missing inputs)."""
    try:
        fn()
    except Exception as e:
        print(f"  SKIP {label}: {e}")

# 1. Per-method cross-dataset comparison (every dataset pair).
for method in INTER_DS_METHODS:
    pairs = [(d, inter_ds_bed(d, method)) for d in ds_list]
    pairs = [(d, p) for d, p in pairs if os.path.exists(p)]
    if len(pairs) < 2:
        print(f"  SKIP inter compare {method}: <2 datasets with this segmentation")
        continue
    segs = [p for _, p in pairs]
    labels = [f"{d}:{method}" for d, _ in pairs]
    bins = [seg_bin(p) for p in segs]
    print(f"Inter-dataset compare: {method} ({len(segs)} datasets)")
    _try(f"compare {method}",
         lambda segs=segs, bins=bins, labels=labels, method=method:
             compare.run_compare(seg=segs, bins=bins, labels=labels, all_pairs=True,
                                 outdir=f"out/{method}"))

# 2. Aggregate per-method kappa matrices into one cross-dataset table.
_try("comparison_table", lambda: compare_out.run_compare_out(
    methods=INTER_DS_METHODS, indir="out",
    outfile="out/comparison_table.tsv"))


In [ ]:
# Pairwise similarity among all ENCODE reference segmentations + reference plots.
ref_segs = sorted(glob.glob(os.path.join(MARKUPS_DIR, "15state", "*.bed.gz")))
if not ref_segs:
    print("No reference markups in", os.path.join(MARKUPS_DIR, "15state"))
else:
    ref_labels = ["_".join(os.path.basename(p).replace(".bed.gz", "").split("_")[1:])
                  for p in ref_segs]
    _try("reference compare", lambda: compare.run_compare(
        seg=ref_segs, bins=CHROMHMM_BIN, labels=ref_labels, all_pairs=True,
        outdir="out/reference"))
    _try("reference summary plots", lambda: summary_plots.run_summary_plots(
        markups_dir=MARKUPS_DIR,
        ref_composition_outfile="out/reference/state_composition.png",
        ref_kappa_matrix="out/reference/kappa_matrix.tsv",
        ref_jaccard_matrix="out/reference/jaccard_similarity_matrix.tsv",
        ref_dist_outfile="out/reference/similarity_distribution.png",
        ref_kappa_noqh_matrix="out/reference/kappa_noqh_matrix.tsv",
        ref_jaccard_noqh_matrix="out/reference/jaccard_noqh_matrix.tsv",
        ref_dist_noqh_outfile="out/reference/similarity_distribution_noqh.png"))


In [ ]:
# Cross-dataset summary bar / violin / distribution plots.
methods_dirs = [f"{d}/methods/{MATCH_METHOD}" for d in ds_list]
analysis_dirs = [f"{d}/analysis/{MATCH_METHOD}" for d in ds_list]

_try("summary bar plots", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, methods_dirs=methods_dirs, analysis_dirs=analysis_dirs,
    methods=INTER_DS_METHODS, outdir=sp_out))

_try("state length violin", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    violin_outfile=f"{sp_out}/state_length_comparison.png"))

_try("state coverage", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    state_coverage_outfile=f"{sp_out}/state_coverage.png"))

_try("peak count", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_count_outfile=f"{sp_out}/peak_count.png"))
_try("peak length", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_length_outfile=f"{sp_out}/peak_length.png"))
_try("peak gap violin", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_gap_violin_outfile=f"{sp_out}/peak_gap_violin.png"))

_try("method similarity distribution", lambda: summary_plots.run_summary_plots(
    method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
    method_sim_dist_outfile=f"{sp_out}/method_similarity_distribution.png",
    method_sim_dist_noqh_outfile=f"{sp_out}/method_similarity_distribution_noqh.png"))

if CHIP_DATASETS and MINT_DATASETS:
    _try("ChIP vs Mint similarity distribution", lambda: summary_plots.run_summary_plots(
        method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
        method_sim_dist_group_a=CHIP_DATASETS, method_sim_dist_group_b=MINT_DATASETS,
        method_sim_dist_filtered_outfile=f"{sp_out}/method_similarity_distribution_chip_vs_mint.png",
        method_sim_dist_filtered_noqh_outfile=f"{sp_out}/method_similarity_distribution_chip_vs_mint_noqh.png"))

if REP_DATASETS:
    _try("replicate consistency", lambda: summary_plots.run_summary_plots(
        datasets=REP_DATASETS, methods_dirs=[f"{d}/methods/comb" for d in REP_DATASETS],
        methods=INTER_DS_METHODS, rep_consistency_outdir=sp_out))

_try("per-dataset state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, methods=INTER_DS_METHODS,
    nstates=NSTATES, match_method=MATCH_METHOD, method_ds_composition_outdir=sp_out))
_try("mean state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    method_composition_outfile=f"{sp_out}/method_state_composition.png"))

# --- Per-assay (ChIP-seq vs Mint-ChIP) splits of the cross-dataset summaries ---
# Regenerate the peaks, total-segments and per-dataset ENCODE-reference plots
# separately for each assay group, into chip/ and mint/ subdirs of sp_out.
for _grp, _gname, _dss in [("chip", "ChIP-seq", CHIP_DATASETS),
                           ("mint", "Mint-ChIP", MINT_DATASETS)]:
    if not _dss:
        continue
    _gdir = f"{sp_out}/{_grp}"
    os.makedirs(_gdir, exist_ok=True)
    _mdirs = [f"{d}/methods/{MATCH_METHOD}" for d in _dss]
    _adirs = [f"{d}/analysis/{MATCH_METHOD}" for d in _dss]
    _try(f"summary bars [{_grp}]", lambda dss=_dss, md=_mdirs, ad=_adirs, gd=_gdir:
         summary_plots.run_summary_plots(datasets=dss, methods_dirs=md, analysis_dirs=ad,
                                         methods=INTER_DS_METHODS, outdir=gd))
    _try(f"peaks [{_grp}]", lambda dss=_dss, gd=_gdir:
         summary_plots.run_summary_plots(datasets=dss, workdir=workdir, methods=INTER_DS_METHODS,
                                         peak_count_outfile=f"{gd}/peak_count.png",
                                         peak_length_outfile=f"{gd}/peak_length.png",
                                         peak_gap_violin_outfile=f"{gd}/peak_gap_violin.png"))
    _try(f"reference n_segments [{_grp}]", lambda dss=_dss, md=_mdirs, gd=_gdir, gn=_gname:
         summary_plots.plot_reference_n_segments(
             dss, md, [DATASETS[d]["cell"] for d in dss],
             f"{gd}/reference_n_segments.png", f"ENCODE reference segments — {gn}"))


In [ ]:
# Emission discriminability (cosine similarity / Gini) per dataset + summary.
analysis_dirs_comb = [f"{d}/analysis/comb" for d in ds_list]
_try("emission similarity", lambda: emission_similarity.run_emission_similarity(
    datasets=ds_list, analysis_dirs=analysis_dirs_comb, methods=INTER_DS_METHODS,
    outdir=sp_out))
_try("binarized emission similarity", lambda: emission_similarity.run_emission_similarity(
    datasets=ds_list, analysis_dirs=analysis_dirs_comb, methods=INTER_DS_METHODS,
    outdir=sp_out,
    out_binem_outfile=f"{sp_out}/out_binem_similarity.png",
    cross_assay_binem_outfile=(f"{sp_out}/cross_assay_binem_similarity.png"
                               if (CHIP_DATASETS and MINT_DATASETS) else None),
    group_a=CHIP_DATASETS, group_b=MINT_DATASETS))


# Results

Every plot produced by the computation cells above, displayed inline and
organised into five sections:

1. **Peaks — number and lengths**
2. **Segmentation — number of states and segments**
3. **Segmentation — state lengths**
4. **Segmentation — state composition**
5. **All other analyses** — entropy, similarity, biological validation,
   emission discriminability, cross-assay portability, replicate consistency
   and per-dataset detail.

Run the helper cell first; missing files are skipped silently, so the output
reflects exactly the segmentations that were produced.

In [ ]:
# ---------------------------------------------------------------------------
# Display helpers. Every plot below was produced by the computation cells above
# and is read straight off disk. Missing files are skipped silently, so the
# notebook renders cleanly regardless of which segmentations were produced.
# ---------------------------------------------------------------------------
VARIANT = MATCH_METHOD                  # default match variant, e.g. "comb"
SP  = "out/summary_plots"     # cross-dataset summary plots
REF = "out/reference"         # ENCODE reference plots

DS_TITLE = {
    "imr90":          "IMR90 (ChIP-seq)",
    "monocytes":      "Monocytes (ChIP-seq)",
    "monocytes_mint": "Monocytes (Mint-ChIP)",
    "gm12878_mint":   "GM12878 (Mint-ChIP)",
    "spleen":         "Spleen (ChIP-seq)",
}

# Per-method segmentations shown in per-dataset grids (de-novo + reference).
METHOD_LABELS = [
    ("ref",              "ENCODE reference"),
    ("chromhmm_default", "Default ChromHMM"),
    ("kmeans_omni",      "KMeans OmniPeak"),
    ("kmeans_homer",     "KMeans HOMER"),
    ("kmeans_macs2",     "KMeans MACS2"),
]
if DO_CHROMHMM_PEAKS:
    METHOD_LABELS += [
        ("chromhmm_omni",  "ChromHMM OmniPeak"),
        ("chromhmm_homer", "ChromHMM HOMER"),
        ("chromhmm_macs2", "ChromHMM MACS2"),
    ]


def header(text, level=3):
    display(HTML(f"<h{level} style='border-bottom:1px solid #999;margin-top:1em'>{text}</h{level}>"))


def show(path, width=820, caption=None):
    """Display one image if it exists; return True when shown."""
    if not os.path.exists(path):
        return False
    if caption:
        display(HTML(f"<div style='color:#555;font-size:0.9em'>{caption}</div>"))
    display(Image(filename=path, width=width))
    return True


def show_group(title, items, width=820, level=3):
    """Show a titled group of plots; the title is skipped when nothing exists.

    Each item is either a path or a (path, caption) tuple.
    """
    norm = [(p, None) if isinstance(p, str) else p for p in items]
    present = [(p, c) for p, c in norm if os.path.exists(p)]
    if not present:
        return False
    header(title, level)
    for p, c in present:
        show(p, width=width, caption=c)
    return True


def show_table(path, caption=None):
    """Display a TSV as a DataFrame if it exists; return True when shown."""
    if not os.path.exists(path):
        return False
    if caption:
        display(HTML(f"<div style='color:#555;font-size:0.9em'>{caption}</div>"))
    display(pd.read_csv(path, sep="\t"))
    return True


def method_plot(ds, method_key, rel):
    """Path to a per-dataset, per-method analysis plot.

    The reference lives outside the variant dir (analysis/ref/...).
    """
    if method_key == "ref":
        return f"{ds}/analysis/ref/{rel}"
    return f"{ds}/analysis/{VARIANT}/{method_key}/{rel}"


print("Display helpers ready (variant:", VARIANT + ").")


## 1. Peaks — number and lengths

Binarization peak statistics: peak count, mean peak length and gap lengths
between adjacent binarized elements, summarised across datasets and shown per
dataset (including replicate Jaccard where replicates exist).

In [ ]:
# 1. Peaks — number and lengths
for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {_gname}", [
        (f"{SP}/{_g}/peak_count.png",      "Peak count per mark and method — mean \u00b1 std across datasets"),
        (f"{SP}/{_g}/peak_length.png",     "Mean peak length per mark and method — mean \u00b1 std across datasets"),
        (f"{SP}/{_g}/peak_gap_violin.png", "Gap lengths between adjacent binarized elements (pooled across datasets)"),
    ], level=2)


## 2. Segmentation — number of states and segments

How fragmented each segmentation is: the total segment count per method across
datasets, the per-reference state/segment counts, and per-dataset
state/segment counts for every segmentation.

In [ ]:
# 2. Segmentation — number of states and segments
for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {_gname}", [
        (f"{SP}/{_g}/summary_n_segments.png", "Total number of segments per method (incl. ENCODE reference) — mean \u00b1 std across datasets"),
    ], level=2)

for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"ENCODE reference segmentations — {_gname}", [
        (f"{SP}/{_g}/reference_n_segments.png", "Number of segments per dataset's ENCODE reference"),
    ])


## 3. Segmentation — state lengths

Segment length distributions: the cross-dataset per-state comparison and
coverage, the ENCODE reference length summaries, and per-dataset segment-length
statistics with per-method length distributions.

In [ ]:
# 3. Segmentation — state lengths
show_group("Cross-dataset summary", [
    (f"{SP}/state_length_comparison.png",  "Per-state segment length: reference vs all de-novo methods"),
    (f"{SP}/state_coverage.png",           "Genomic coverage fraction per chromatin state"),
    (f"{SP}/summary_mean_tx_length.png",   "Mean Tx (transcription) segment length"),
], level=2)

show_group("ENCODE reference segmentations", [
    (f"{REF}/mean_length.png",   "Mean segment length"),
    (f"{REF}/median_length.png", "Median segment length"),
    (f"{REF}/min_length.png",    "Min segment length"),
    (f"{REF}/max_length.png",    "Max segment length"),
], width=750)


## 4. Segmentation — state composition

Fraction of the genome covered by each chromatin state: averaged per method
across datasets, across the ENCODE reference segmentations, and per dataset for
each de-novo method.

In [ ]:
# 4. Segmentation — state composition
show_group("Per-method composition (mean across datasets)", [
    (f"{SP}/method_state_composition.png", "State composition per method — mean across datasets"),
], level=2)

show_group("ENCODE reference composition", [
    (f"{REF}/state_composition.png", "State composition across ENCODE reference segmentations"),
])

comp = [(p, os.path.basename(p).replace("method_ds_composition_", "").replace(".png", ""))
        for p in sorted(glob.glob(f"{SP}/method_ds_composition_*.png"))]
show_group("Per-dataset composition for each de-novo method", comp)


## 5. All other analyses

Transition entropy, inter-dataset / inter-reference similarity, biological
validation (RNA-seq / ATAC-seq), emission discriminability, cross-assay
portability, replicate consistency, the aggregated comparison table, and the
per-dataset detail (method comparison tables, functional enrichment and state
emissions for each method).

In [ ]:
# 5. All other analyses
show_group("Transition matrix entropy", [
    (f"{SP}/summary_entropy.png",           "De-novo — Full (raw labels)"),
    (f"{SP}/summary_entropy_noqh.png",      "De-novo — NOQH (excl. Quies/Het)"),
    (f"{REF}/entropy_summary_combined.png", "ENCODE reference entropy (Full + NOQH)"),
], level=2)

show_group("Segmentation similarity", [
    (f"{SP}/method_similarity_distribution.png",      "De-novo inter-dataset similarity — Full"),
    (f"{SP}/method_similarity_distribution_noqh.png", "De-novo inter-dataset similarity — NOQH"),
    (f"{REF}/similarity_distribution.png",            "Inter-reference similarity — Full"),
    (f"{REF}/similarity_distribution_noqh.png",       "Inter-reference similarity — NOQH"),
    (f"{SP}/out_binem_similarity.png",      "Inter-dataset binarized emission cosine similarity"),
    (f"{SP}/per_state_kappa_summary.png",             "Mean per-state Cohen's Kappa vs ENCODE reference"),
], level=2)

show_group("Biological validation (RNA-seq / ATAC-seq)", [
    (f"{SP}/summary_jaccard_tx.png",        "Jaccard: Tx state vs expressed gene bodies"),
    (f"{SP}/summary_enrich_tx.png",         "Tx fold enrichment at expressed gene bodies"),
    (f"{SP}/summary_jaccard_tss.png",       "Jaccard: Tss state vs RefSeq TSS ±1 kb"),
    (f"{SP}/summary_jaccard_tss_atac.png",  "Jaccard: Tss state vs ATAC-seq peaks"),
], level=2)

show_group("Emission discriminability (cosine similarity / Gini)", [
    (f"{SP}/emission_cosine_sim_summary.png", "Pairwise cosine similarity of state emissions"),
    (f"{SP}/emission_gini_summary.png",       "Gini index of state emissions"),
], level=2)

show_group("Cross-assay portability (ChIP ↔ Mint-ChIP)", [
    (f"{SP}/method_similarity_distribution_chip_vs_mint.png",      "Full"),
    (f"{SP}/method_similarity_distribution_chip_vs_mint_noqh.png", "NOQH"),
    (f"{SP}/cross_assay_binem_similarity.png",                     "Cross-assay binarized emission similarity"),
], level=2)

show_group("Replicate consistency", sorted(glob.glob(f"{SP}/rep_consistency_*.png")), level=2)

header("Cross-dataset comparison table", 2)
if not show_table("out/comparison_table.tsv"):
    print("  (no aggregated comparison table)")



## 6. Per-state matching matrices

Work-state → ENCODE-reference matching score matrices produced by `match.py` (combined overlap+emission, `comb` variant). Rows are the de-novo method's states, columns the reference states; the Hungarian-selected match per row is outlined in red. Produced by the pipeline as `{...}_comb_matched.match.png` next to each matched BED.

In [ ]:
# Per-state matching matrices (work → ENCODE reference), comb variant.
header("Per-state matching matrices (work → ENCODE reference)", 2)
_match_methods = [("chromhmm_default", "Default ChromHMM"),
                  ("kmeans_omni",      "KMeans OmniPeak"),
                  ("kmeans_homer",     "KMeans HOMER"),
                  ("kmeans_macs2",     "KMeans MACS2")]
for ds in DATASETS:
    items = [(inter_ds_bed(ds, k).replace(".bed", ".match.png"),
              f"{lbl}: per-state matching score (matched cell outlined in red)")
             for k, lbl in _match_methods]
    show_group(DS_TITLE.get(ds, ds), items, width=620, level=3)


# 7. All other per dataset plots

In [ ]:
for ds in DATASETS:
    show_group(DS_TITLE.get(ds, ds), [
        (f"{ds}/peaks/n_peaks.png",              "Number of peaks per mark"),
        (f"{ds}/peaks/mean_length.png",          "Mean peak length per mark"),
        (f"{ds}/peaks/median_length.png",        "Median peak length per mark"),
        (f"{ds}/peaks/jaccard_rep1_vs_rep2.png", "Peak Jaccard: rep1 vs rep2"),
    ], width=600, level=3)


    show_group(DS_TITLE.get(ds, ds), [
        (f"{ds}/comparison/{VARIANT}/n_segments.png",      "Number of segments per segmentation"),
        (f"{ds}/comparison/{VARIANT}/mean_Tx_length.png",  "Mean Tx length per segmentation"),
        (f"{SP}/per_state_kappa_{ds}.png",                 "Per-state Cohen's Kappa vs ENCODE reference"),
    ], width=750, level=3)

    show_group(DS_TITLE.get(ds, ds), [
        (f"{ds}/comparison/{VARIANT}/mean_length.png",   "Mean segment length per segmentation"),
        (f"{ds}/comparison/{VARIANT}/median_length.png", "Median segment length per segmentation"),
        (f"{ds}/comparison/{VARIANT}/min_length.png",    "Min segment length per segmentation"),
        (f"{ds}/comparison/{VARIANT}/max_length.png",    "Max segment length per segmentation"),
    ], width=750, level=3)
    show_group(f"{DS_TITLE.get(ds, ds)} — per-method length distribution",
               [(method_plot(ds, k, "segment_length.png"), lbl) for k, lbl in METHOD_LABELS],
               width=600, level=4)


    header(DS_TITLE.get(ds, ds), 3)
    show_table(f"{ds}/methods/{VARIANT}/comparison_table.tsv", caption="Method comparison table")
    for k, lbl in METHOD_LABELS:
        show_group(lbl, [
            (method_plot(ds, k, "enrichment/enrichment.png"),         f"{lbl}: functional enrichment"),
            (method_plot(ds, k, "bin_emissions/state_emissions.png"), f"{lbl}: binarized emissions"),
            (method_plot(ds, k, "bw_emissions/state_emissions.png"),  f"{lbl}: bigwig emissions"),
        ], width=760, level=4)
